# Vision Transformer: five-fold cross-validation

A transformer member for the CNN ensemble (ConvNeXt, EfficientNetV2-S, BiT, HRNet). The backbone is the winner of the
frozen screen in [22_vit_frozen_screen.ipynb](../22_vit_frozen_screen.ipynb); `STAGE2_VIT` picks one of the recipes in the
next cell (default `dinov2_b`). The per-fold loop, records, thresholds and submissions are the same as in
`05_cv_hrnet_w32.ipynb`, so the OOF predictions can be compared and averaged with the CNNs image by image.

**What differs from the CNN recipes, and why:**
- **Input (height × width).** The photos are 450×375. Patch-14 ViTs get **448×378**: the whole photo at about native
  scale with ~5 px of padding, instead of the 384×320 photo inside a 384² square (17% padding) that the CNNs see.
  timm resamples the pretrained position embeddings to this grid (32×27 patches). SigLIP (patch 16) gets 448×368;
  Swin keeps its native 384² square because of its 12×12 windows.
- **Layer-wise LR decay.** The backbone LR is the LR of the top block and shrinks by the factor `layer_decay` per block
  towards the patch embedding, so the lowest, most generic layers barely move. The new head keeps its own LR.
- **Weight decay 0.05**, but none on biases, norms, the position embedding, the class and register tokens.
- **Stochastic depth** (`drop_path_rate` 0.1).
- **No BatchNorm**, so nothing to freeze and the physical batch does not change the statistics.

Everything else is unchanged: flip augmentation, BCEWithLogitsLoss, AdamW, one warm-up epoch then cosine to 10%,
effective batch 16, clip 1.0, head warm-up on cached frozen features, selection by held-out macro F1 at 0.25.
These hyperparameters are standard ViT fine-tuning values, not tuned on this data.

**There is no earlier fold-0 run to import**, so all five folds are trained here.

**Resume, records and flags** work as in notebook 05: `STAGE2_RUN_TRAINING=1` to train, `STAGE2_CV_FOLDS=1,2` to split
sessions, `STAGE2_SMOKE_TEST=1` for a few batches into `<key>_smoke/`, `STAGE2_CV_OUTPUT` for a Drive folder on Colab.
Measure one epoch before planning the five folds (`STAGE2_CV_FOLDS=0` and stop after the first epoch line).


## Colab setup (skipped elsewhere)

On Colab this cell mounts Drive, unpacks the code and data from `MyDrive/AI301-stage2/vit_input/` (made by `scripts/stage_vit_colab_inputs.py`) onto the VM disk, installs the missing packages, and points all results at Drive:

- `STAGE2_CV_OUTPUT` → `MyDrive/AI301-stage2/cv/` (fold records, checkpoints, OOF and test predictions, `cv.json`), so every epoch survives a VM reset;
- `submissions/` on the VM is a link to `MyDrive/AI301-stage2/submissions/`, so the CSVs written at the end are not lost.

Edit `SESSION` below for each session. The first session should be a smoke test (`STAGE2_SMOKE_TEST='1'`), the second one epoch of fold 0 to measure the time, and then the real folds. Any variable can also be set beforehand with `os.environ`; the GPU-dependent precision and batch size are only defaults.


In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = importlib.util.find_spec('google.colab') is not None
SESSION = {
    'STAGE2_VIT': 'dinov2_b',
    'STAGE2_RUN_TRAINING': '1',
    'STAGE2_SMOKE_TEST': '1',   # '1' = a few batches per fold into dinov2_b_smoke/; set '0' for the real run
    'STAGE2_CV_FOLDS': '0',     # folds of this session, e.g. '0', '1,2', '3,4'
    'STAGE2_USE_WANDB': '0',
    'STAGE2_ALLOW_WEIGHT_DOWNLOAD': '1',  # Colab has no weight cache; the first run downloads from the Hugging Face hub
}
if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/AI301-stage2')
    INPUT = DRIVE / 'vit_input'
    WORK = Path('/content/ai301')
    if not (INPUT / 'code' / 'stage2' / 'cv.py').is_file():
        raise FileNotFoundError(f'Upload the staged inputs to {INPUT} first (scripts/stage_vit_colab_inputs.py).')
    shutil.copytree(INPUT / 'code', WORK, dirs_exist_ok=True)  # code is refreshed every session
    if not (WORK / 'data' / 'test').is_dir():  # images only once per VM
        archive = INPUT / 'data.zip'
        if archive.is_file():
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(WORK)
        else:
            fallback = DRIVE / 'hrnet_w32_input' / 'data'  # staged for the HRNet run, same files, loose
            shutil.copytree(fallback, WORK / 'data')
    counts = {s: len(list((WORK / 'data' / s).glob('*.jpg'))) for s in ('train', 'test')}
    if counts != {'train': 12260, 'test': 18673}:
        raise ValueError(f'Unexpected image counts {counts}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightning>=2.5,<3', 'timm>=1.0,<2', 'scikit-learn>=1.5,<2'], check=True)
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('No GPU. Runtime > Change runtime type > pick a GPU (A100 or L4 if you have them, else T4).')
    ampere = torch.cuda.get_device_capability()[0] >= 8
    print('GPU:', torch.cuda.get_device_name(), '| bf16' if ampere else '| fp16')
    (DRIVE / 'submissions').mkdir(parents=True, exist_ok=True)
    if not (WORK / 'submissions').exists():
        (WORK / 'submissions').symlink_to(DRIVE / 'submissions')
    os.environ.update(SESSION)
    os.environ['STAGE2_ROOT'] = str(WORK)
    os.environ['STAGE2_CV_OUTPUT'] = str(DRIVE / 'cv')
    os.environ.setdefault('STAGE2_PRECISION', 'bf16-mixed' if ampere else '16-mixed')  # T4 has no bf16
    os.environ.setdefault('STAGE2_BATCH_SIZE', '8' if ampere else '4')
    os.environ.setdefault('STAGE2_NUM_WORKERS', str(min(8, os.cpu_count() or 2)))
    os.chdir(WORK)
    print({k: os.environ[k] for k in (*SESSION, 'STAGE2_PRECISION', 'STAGE2_BATCH_SIZE', 'STAGE2_NUM_WORKERS', 'STAGE2_CV_OUTPUT')})
else:
    print('Not on Colab: nothing to set up.')


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import numpy as np
import torch

RUN_TRAINING = os.environ.get('STAGE2_RUN_TRAINING', '0') == '1'
SMOKE_TEST = os.environ.get('STAGE2_SMOKE_TEST', '0') == '1'  # A few batches per fold; results go to a separate *_smoke folder.
FOLDS = [int(fold) for fold in os.environ.get('STAGE2_CV_FOLDS', '0,1,2,3,4').split(',')]  # Folds this session may train.
REUSE_FOLD0 = False  # Nothing to import for a new model family.
ALLOW_WEIGHT_DOWNLOAD = os.environ.get('STAGE2_ALLOW_WEIGHT_DOWNLOAD', '0') == '1'
USE_WANDB = os.environ.get('STAGE2_USE_WANDB', '1') == '1'
REMOVE_RUNTIME_ENV_AFTER_LOAD = os.environ.get('STAGE2_REMOVE_RUNTIME_ENV_AFTER_LOAD', '0') == '1'
CV_OUTPUT = os.environ.get('STAGE2_CV_OUTPUT')  # e.g. /content/drive/MyDrive/AI301-stage2/cv on Colab; default experiments/cv.
BUILD_SUBMISSION = True  # Writes two CSVs once all five folds are complete; existing files are kept.
SUBMIT_TO_KAGGLE = False  # No submission API call exists in this notebook.

if not ALLOW_WEIGHT_DOWNLOAD:
    os.environ['HF_HUB_OFFLINE'] = '1'  # Must be set before anything imports huggingface_hub.

start = Path(os.environ.get('STAGE2_ROOT', Path.cwd())).resolve()
ROOT = next((path for path in (start, *start.parents) if (path / 'folds.csv').is_file() and (path / 'stage2' / 'cv.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter from this repository or a subdirectory, or set STAGE2_ROOT.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from stage2.cv import Recipe, Runtime

VIT_RECIPES = {
    # (key, title, timm model, input (height, width) or None for the 384 square, layer decay)
    key: Recipe(
        key=key, title=title, model_name=model_name,
        backbone_lr=2e-5, head_lr=2e-4, weight_decay=0.05, max_epochs=8, patience=3,
        image_hw=image_hw, model_kwargs=(('drop_path_rate', 0.1),), layer_decay=layer_decay,
        head_epochs=10, head_patience=3,
    )
    for key, title, model_name, image_hw, layer_decay in (
        ('dinov2_b', 'DINOv2 ViT-B/14 + registers', 'vit_base_patch14_reg4_dinov2.lvd142m', (378, 448), 0.75),
        ('dinov2_s', 'DINOv2 ViT-S/14 + registers', 'vit_small_patch14_reg4_dinov2.lvd142m', (378, 448), 0.75),
        ('eva02_b', 'EVA-02 ViT-B/14', 'eva02_base_patch14_448.mim_in22k_ft_in22k_in1k', (378, 448), 0.75),
        ('siglip_b', 'SigLIP ViT-B/16', 'vit_base_patch16_siglip_384.webli', (368, 448), 0.75),
        ('swin_b', 'Swin-B (window 12)', 'swin_base_patch4_window12_384.ms_in22k_ft_in1k', None, 0.9),
    )
}
RECIPE = VIT_RECIPES[os.environ.get('STAGE2_VIT', 'dinov2_b')]
FOLD0_SOURCE = None  # No earlier fold-0 run of a transformer exists.

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
RUNTIME = Runtime(
    batch_size=int(os.environ.get('STAGE2_BATCH_SIZE', '4' if DEVICE == 'cuda' else '2')),  # Physical batch; accumulation keeps the effective batch at 16.
    precision=os.environ.get('STAGE2_PRECISION', '16-mixed' if DEVICE == 'cuda' else '32-true'),
    num_workers=int(os.environ.get('STAGE2_NUM_WORKERS', '2' if DEVICE == 'cuda' else '0')),
    device=DEVICE, smoke_test=SMOKE_TEST, use_wandb=USE_WANDB and RUN_TRAINING and not SMOKE_TEST,
)
if RECIPE.effective_batch_size % RUNTIME.batch_size:
    raise ValueError('The effective batch size must be divisible by the physical batch size.')
if not set(FOLDS) <= set(range(5)):
    raise ValueError('STAGE2_CV_FOLDS must list folds 0-4.')
if RUN_TRAINING and DEVICE == 'cpu' and not SMOKE_TEST:
    raise RuntimeError('No GPU found. A CPU run would take days; use a CUDA or Apple GPU.')
print(f'{RECIPE.title} | device {DEVICE} | {RUNTIME.precision} | batch {RUNTIME.batch_size} x {RECIPE.effective_batch_size // RUNTIME.batch_size} accumulation')
print(f'training: {RUN_TRAINING}; smoke test: {SMOKE_TEST}; folds this session: {FOLDS}; reuse fold 0: {REUSE_FOLD0}; W&B: {RUNTIME.use_wandb}')

## Same split as before

We check that `folds.csv` is still the original seed-42 split, stratified by label combination, and join it to the labels by image ID. Every model in this folder uses the same five folds, so their OOF predictions can be compared and averaged image by image. The table shows how many named-pet photos each held-out fold has, and which folds are already done.

In [ ]:
from stage2.cv import collect_predictions, epoch_history, ensure_wandb_key, fold_dir, fold_status, import_fold, run_fold, save_cv_summary
from stage2.experiment import LABEL_COLUMNS, form_submission, form_submission_per_label, preflight_images, read_csv, score_probabilities, split_rows, verify_saved_stratification, write_submission
from stage2.thresholds import nested_thresholds, per_label_f1

DATA_DIR = ROOT / 'data'
verify_saved_stratification(DATA_DIR, ROOT / 'folds.csv', RECIPE.seed)
train_rows = read_csv(DATA_DIR / 'train.csv', ('image_id', *LABEL_COLUMNS))
fold_by_id = {row['image_id']: int(row['fold']) for row in read_csv(ROOT / 'folds.csv', ('image_id', 'fold'))}
test_rows = read_csv(DATA_DIR / 'test.csv', ('image_id',))
test_ids = [row['image_id'] for row in test_rows]
CV_ROOT = Path(CV_OUTPUT) if CV_OUTPUT else ROOT / 'experiments' / 'cv'
MODEL_DIR = CV_ROOT / (RECIPE.key + ('_smoke' if SMOKE_TEST else ''))
print('Results folder:', MODEL_DIR)


def show_status() -> None:
    print(f"{'fold':<5} {'images':>7} {'Moris':>6} {'Motya':>6} {'Biatrix':>8}  status")
    for fold in range(5):
        held_out = [row for row in train_rows if fold_by_id[row['image_id']] == fold]
        counts = [sum(int(row[label]) for row in held_out) for label in ('Moris', 'Motya', 'Biatrix')]
        print(f'{fold:<5} {len(held_out):>7} {counts[0]:>6} {counts[1]:>6} {counts[2]:>8}  {fold_status(MODEL_DIR, fold)}')


show_status()

## Train the folds

For each fold of this session, in order:

1. **Skip** it if its `fold.json` exists and all its files match their recorded hashes. If the record exists but a file has changed, stop and leave it as it is.
2. **Fold 0:** import the earlier run if `REUSE_FOLD0` is set.
3. **Phase 1:** train a new linear head on the frozen backbone, on the training folds of this fold only (skipped if `head/head.json` exists).
4. **Phase 2:** fine-tune the whole network. `checkpoints/last.ckpt` holds the full state (weights, optimizer, scheduler, early stopping) after every epoch, and an interrupted fold continues from it. `training_done.json` marks the end of training, after which the selected checkpoint is never trained further.
5. **Predict** the held-out fold and the 18,673 test images with the selected checkpoint. We save probabilities and logits, then `fold.json` with the metrics and the hash of every file.

The images are decoded once before the first fold, so a damaged file fails early. Fine-tuning logs to W&B (one run per fold, grouped by model) and to a CSV log in the fold folder.

In [ ]:
if RUN_TRAINING:
    if RUNTIME.use_wandb:
        ensure_wandb_key(ROOT, REMOVE_RUNTIME_ENV_AFTER_LOAD)
        print('W&B online credential is available (value hidden).')
    pending = [fold for fold in FOLDS if fold_status(MODEL_DIR, fold) != 'completed']
    broken = [fold for fold in pending if fold_status(MODEL_DIR, fold) == 'broken']
    if broken:
        raise RuntimeError(f'Folds {broken} have a fold.json whose files changed. Inspect them; nothing was overwritten.')
    if pending and not SMOKE_TEST:
        preflight_images(train_rows, DATA_DIR / 'train')
        preflight_images(test_rows, DATA_DIR / 'test')
    for fold in pending:
        train_part, validation_part = split_rows(DATA_DIR, ROOT / 'folds.csv', fold)
        started = time.perf_counter()
        if fold == 0 and REUSE_FOLD0 and not SMOKE_TEST and fold_status(MODEL_DIR, 0) == 'not started':
            import_fold(RECIPE, ROOT, MODEL_DIR, 0, *FOLD0_SOURCE, validation_part, test_ids)
            continue
        print(f'--- fold {fold}: train {len(train_part)} / held out {len(validation_part)} images ---')
        run_fold(RECIPE, RUNTIME, ROOT, MODEL_DIR, fold, train_part, validation_part, test_rows)
        print(f'Fold {fold} took {(time.perf_counter() - started) / 3600:.2f} h in this session.')
    show_status()
else:
    print('Training disabled; set STAGE2_RUN_TRAINING=1 to train the folds of this session.')

## Learning curves of every fold

Training and validation loss and held-out macro F1 per epoch, one line per trained fold. A resumed fold continues the same line. An imported fold 0 has its curves in the earlier run's folder. We expect what fold 0 showed: training loss near zero, and validation loss flat or rising after a few epochs. The F1 curve moves in steps of single named-pet photos (about 0.012 macro F1 per photo, as on fold 0). The figure is saved as `curves.png` in the model folder.

In [ ]:
import matplotlib.pyplot as plt

histories = {fold: epoch_history(fold_dir(MODEL_DIR, fold)) for fold in range(5) if (fold_dir(MODEL_DIR, fold) / 'curves').is_dir()}
histories = {fold: rows for fold, rows in histories.items() if rows}
if histories:
    figure, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for fold, rows in histories.items():
        for axis, name in zip(axes, ('train_loss', 'val_loss', 'val_macro_f1')):
            points = [(row['epoch'], row[name]) for row in rows if name in row]
            if points:
                axis.plot(*zip(*points), marker='o', label=f'fold {fold}')
        best = max((row for row in rows if 'val_macro_f1' in row), key=lambda row: row['val_macro_f1'])
        print(f"fold {fold}: {len(rows)} epochs, best held-out macro F1 {best['val_macro_f1']:.5f} at epoch {best['epoch']}")
    for axis, title in zip(axes, ('Training loss', 'Validation loss', 'Held-out macro F1 at 0.25')):
        axis.set(title=title, xlabel='Epoch')
        axis.legend()
    axes[0].set_yscale('log')
    figure.suptitle(RECIPE.title)
    figure.tight_layout()
    figure.savefig(MODEL_DIR / 'curves.png', dpi=140)
    plt.show()
else:
    print('No trained fold has curves yet.')

## Cross-validation score at the fixed threshold 0.25

This needs all five folds. For each fold we report the held-out macro F1 at 0.25 and per-label F1 and AP. Below the table come the mean and standard deviation over the folds, and the pooled OOF score, which is macro F1 over all 12,260 images at once. The pooled score is the one to compare between models. The fold scores show how much one fold can move. Compare the pooled OOF score with the CNNs in `experiments/cv/*/cv.json` (HRNet 0.98613, ConvNeXt 0.97416 at 0.25).

The OOF scores are still a little optimistic, because each fold's checkpoint was selected on that fold. That bias is the same for every model in this folder, so the comparison between models stays fair.

In [ ]:
completed = [fold for fold in range(5) if fold_status(MODEL_DIR, fold) == 'completed']
CV_READY = len(completed) == 5 and not SMOKE_TEST
if not CV_READY:
    print(f'The CV results need all five folds; completed: {completed}.' + (' A smoke run has no results.' if SMOKE_TEST else ''))
else:
    cv = collect_predictions(MODEL_DIR, train_rows, fold_by_id, test_ids)
    print(f"{'fold':<5} {'source':<9} {'macro F1':>9} " + ' '.join(f'{label:>8}' for label in LABEL_COLUMNS) + f" {'macro AP':>9}")
    fold_scores = {}
    for fold in range(5):
        held_out = cv.folds == fold
        scores = score_probabilities(cv.targets[held_out], cv.oof[held_out], RECIPE.threshold)
        fold_scores[fold] = scores
        print(f"{fold:<5} {cv.records[fold]['source']:<9} {scores['macro_f1']:>9.5f} " + ' '.join(f"{scores['per_label_f1'][label]:>8.4f}" for label in LABEL_COLUMNS) + f" {scores['macro_ap']:>9.5f}")
    values = np.array([scores['macro_f1'] for scores in fold_scores.values()])
    pooled = score_probabilities(cv.targets, cv.oof, RECIPE.threshold)
    pooled_05 = score_probabilities(cv.targets, cv.oof, 0.5)
    print(f'\nMean over folds: {values.mean():.5f} +- {values.std(ddof=1):.5f} (min {values.min():.5f}, max {values.max():.5f})')
    print(f"Pooled OOF macro F1 at 0.25: {pooled['macro_f1']:.5f}; at 0.5: {pooled_05['macro_f1']:.5f}; macro AP {pooled['macro_ap']:.5f}")
    decided = cv.oof >= RECIPE.threshold
    print('\nOOF mistakes at 0.25 (missed / false alarms):',
          {label: (int((~decided & (cv.targets == 1))[:, i].sum()), int((decided & (cv.targets == 0))[:, i].sum())) for i, label in enumerate(LABEL_COLUMNS)})

## One threshold per label

On the OOF predictions every label gets its own threshold: the middle of the range of thresholds that reach that label's best F1, on a grid from 0.02 to 0.98 in steps of 0.01. The middle of the range is less sensitive to a single photo than either end.

If we choose a threshold on the same predictions we score, the score is optimistic. We therefore also report a **fair** score: for every fold, the thresholds are chosen on the other four folds only and then applied to that fold, and the five held-out folds are scored together. The table lists the five threshold sets as well, so we can see how stable the choice is. The test set uses the thresholds chosen on all five folds.

Read the result with the noise level in mind. With 41–56 photos per named pet, one photo moves that label's F1 by about 0.01 and macro F1 by about 0.002–0.0025. A fair gain below about 0.005 (two photos) is not a real gain.

In [ ]:
if CV_READY:
    tuning = nested_thresholds(cv.targets, cv.oof, cv.folds)
    thresholds = np.array([tuning['thresholds'][label] for label in LABEL_COLUMNS])
    fixed = per_label_f1(cv.targets, cv.oof, RECIPE.threshold)
    print(f"{'label':<9}{'photos':>7}{'F1 @0.25':>10}{'threshold':>11}  {'chosen on 4 folds (held-out fold 0..4)':<42}{'F1 tuned':>9}{'F1 fair':>9}")
    for index, label in enumerate(LABEL_COLUMNS):
        by_fold = ' '.join(f"{tuning['thresholds_by_held_out_fold'][fold][label]:.2f}" for fold in range(5))
        print(f"{label:<9}{int(cv.targets[:, index].sum()):>7}{fixed[index]:>10.4f}{thresholds[index]:>11.2f}  {by_fold:<42}"
              f"{tuning['optimistic_per_label_f1'][label]:>9.4f}{tuning['fair_per_label_f1'][label]:>9.4f}")
    print(f"{'macro':<9}{'':>7}{fixed.mean():>10.5f}{'':>11}  {'':<42}{tuning['optimistic_macro_f1']:>9.5f}{tuning['fair_macro_f1']:>9.5f}")
    print(f"\nFair gain of per-label thresholds over 0.25: {tuning['fair_macro_f1'] - fixed.mean():+.5f} macro F1")

## The fold ensemble on the test set

The test prediction of this model is the equal-weight mean of the five fold models' probabilities. We compare it with the fold-0 model alone. The counts show how many test photos each version calls a named pet, and how many test cells the fold models disagree on. A cell is one image-label pair. A cell on which the fold models disagree is decided by the average. The test labels are hidden, so this shows how much the fold ensemble changes, not whether the changes are right.

One caveat for the tuned thresholds: an average of five models is a little less extreme than one model, so a threshold tuned on single-model OOF probabilities may fit the fold mean slightly differently. The leaderboard shows whether that matters.

In [ ]:
if CV_READY:
    test_mean = cv.test_mean
    single = cv.test_by_fold[0]
    named = [LABEL_COLUMNS.index(label) for label in ('Moris', 'Motya', 'Biatrix')]
    print(f"{'test positives':<34}" + ''.join(f'{label:>9}' for label in LABEL_COLUMNS))
    for title, decided in (
        ('fold-0 model alone, 0.25', single >= RECIPE.threshold),
        *((f'fold-{fold} model alone, 0.25', cv.test_by_fold[fold] >= RECIPE.threshold) for fold in range(1, 5)),
        ('fold ensemble, 0.25', test_mean >= RECIPE.threshold),
        ('fold ensemble, per-label thresholds', test_mean >= thresholds),
    ):
        print(f'{title:<34}' + ''.join(f'{int(decided[:, index].sum()):>9}' for index in range(len(LABEL_COLUMNS))))
    votes = np.stack([cv.test_by_fold[fold] >= RECIPE.threshold for fold in range(5)]).sum(axis=0)
    split = (votes > 0) & (votes < 5)
    print(f'\nCells where the five fold models disagree at 0.25: {int(split.sum())} ({int(split[:, named].sum())} named-pet cells)')
    changed = (test_mean >= RECIPE.threshold) != (single >= RECIPE.threshold)
    print(f'Fold ensemble vs the fold-0 model alone, both at 0.25: {int(changed.sum())} cells change ({int(changed[:, named].sum())} named-pet)')
    changed = (test_mean >= thresholds) != (test_mean >= RECIPE.threshold)
    print(f'Per-label thresholds vs 0.25 on the fold ensemble: {int(changed.sum())} cells change ({int(changed[:, named].sum())} named-pet)')

## Save the CV result and the submission files

`cv.json` in the model folder holds the CV scores, the thresholds and the hashes of the five fold records it was built from. `oof.csv` (12,260 rows with their fold) and `test_mean.csv` (the fold ensemble) are the inputs of `06_cv_ensemble.ipynb`. Two CSVs go to `submissions/`: `cv_<key>_5fold_025.csv` (threshold 0.25) and `cv_<key>_5fold_perlabel.csv` (per-label thresholds). Submitting both separates the effect of the fold ensemble from the effect of the thresholds. Nothing is overwritten. Upload by hand and add the public scores to `public_leaderboard_score` in `cv.json` and to the table below.

In [ ]:
if CV_READY:
    summary = {
        'cv': {
            'threshold': RECIPE.threshold,
            'per_fold_macro_f1': {str(fold): scores['macro_f1'] for fold, scores in fold_scores.items()},
            'per_fold_per_label_f1': {str(fold): scores['per_label_f1'] for fold, scores in fold_scores.items()},
            'mean_fold_macro_f1': float(values.mean()), 'std_fold_macro_f1': float(values.std(ddof=1)),
            'pooled_oof': pooled, 'pooled_oof_macro_f1_at_0_5': pooled_05['macro_f1'],
        },
        'per_label_thresholds': tuning,
        'test': {'method': 'equal-weight mean of the five fold models',
                 'positives_at_0_25': dict(zip(LABEL_COLUMNS, map(int, (test_mean >= RECIPE.threshold).sum(axis=0)))),
                 'positives_per_label_thresholds': dict(zip(LABEL_COLUMNS, map(int, (test_mean >= thresholds).sum(axis=0))))},
    }
    record = save_cv_summary(MODEL_DIR, ROOT, cv, RECIPE, summary)
    print('CV record:', MODEL_DIR / 'cv.json')
    if BUILD_SUBMISSION:
        sample = DATA_DIR / 'sample_submission.csv'
        for suffix, rows in (('025', form_submission(sample, test_ids, test_mean, RECIPE.threshold)),
                             ('perlabel', form_submission_per_label(sample, test_ids, test_mean, thresholds))):
            path = ROOT / 'submissions' / f'cv_{RECIPE.key}_5fold_{suffix}.csv'
            if path.exists():
                print('Kept existing', path.relative_to(ROOT))
            else:
                write_submission(path, rows)
                print('Wrote', path.relative_to(ROOT), {label: sum(row[label] for row in rows) for label in LABEL_COLUMNS})
else:
    print('Nothing saved yet; the CV summary needs all five folds.')

## Results and conclusions

_DINOv2-B ViT-B/14 (`dinov2_b`), five folds. CV numbers are from `experiments/cv/dinov2_b/cv.json`; public scores are from the Kaggle submissions page. Not written into `cv.json`, because the ensemble records store its hash._

| Version | CV macro F1 | Public LB |
|---|---:|---:|
| Five folds: mean ± std of the fold scores, 0.25 | 0.98735 ± 0.00764 | — |
| Five folds: pooled OOF, 0.25 (`cv_<key>_5fold_025.csv`) | 0.98711 | **0.97828** |
| Five folds: per-label thresholds, fair (`cv_<key>_5fold_perlabel.csv`) | 0.98356 | 0.97460 |

Questions to answer here:

1. **Does the transformer beat the best CNN on its own?** Compare pooled OOF macro F1 and macro AP with HRNet (0.98613 / 0.99283).
2. **Does it make different mistakes?** Count OOF and test named-pet cells where it disagrees with the CNN ensemble.
3. **Does it improve the ensemble?** Add it as a member in `06_cv_ensemble.ipynb` and compare the OOF macro F1 with the bootstrap there.


The fixed 0.25 threshold scored higher on the public set than the per-label thresholds (0.97828 against 0.97460), the same order as for the CNNs. For comparison, the best earlier CNN ensembles scored 0.98367 and 0.98374 (notebook 06).